# HW1: CIFAR10 Speed and Accuracy Challenge

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/HW1-CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

CIFAR10 is a dataset of 32x32 color images in ten classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

https://www.cs.toronto.edu/~kriz/cifar.html

**Goal: get the highest possible test accuracy with at most 10 minutes of training time on a Colab T4.**

Rules
* Train only on the CIFAR10 training data. **Do not use the test set for training or hyperparameter tuning** (you may split a validation set off the training data yourself)
* No pretrained models (train from scratch)
* Training time (the run time of `train()` below) must not exceed 10 minutes (on a T4)
* Submit: the notebook (with its outputs)

Things to try
* Model architecture: wider/deeper ResNets, different normalizations and activations
* Data augmentation: random crop, flip, cutout, mixup, ...
* Optimization: optimizer, learning rate schedule, batch size, label smoothing, weight decay
* Speed: mixed precision (fp16), `channels_last`, `torch.compile`, keeping the data on the GPU
* Test time augmentation

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Download from the Hugging Face Hub (much faster than the original CIFAR site)
from datasets import load_dataset
from types import SimpleNamespace
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    return SimpleNamespace(data=np.stack([np.array(im) for im in ds["img"]]), targets=list(ds["label"]))
train_raw = load_cifar10("train")
test_raw = load_cifar10("test")
name_array = np.array(["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"])

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def to_tensor(raw):
    X = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)
    return ((X - mean) / std).to(device), torch.tensor(raw.targets).to(device)
train_X, train_y = to_tensor(train_raw)
test_X, test_y = to_tensor(test_raw)
train_X.shape, test_X.shape

In [ ]:
def showX(X):
    display(Image.fromarray(np.concatenate(list(X), axis=1)).resize((64 * len(X), 64), Image.NEAREST))
showX(train_raw.data[:12])
print(name_array[train_raw.targets[:12]])

## Your code

Complete `train()`, which returns the trained model, and `predict(model, X)`, which returns the predicted class for each image.

In [ ]:
def train():
    # TODO
    model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(5):
        perm = torch.randperm(len(train_X), device=device)
        for i in range(0, len(train_X), 256):
            idx = perm[i:i + 256]
            loss = F.cross_entropy(model(train_X[idx]), train_y[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(epoch, loss.item())
    return model

@torch.no_grad()
def predict(model, X):
    # TODO
    model.eval()
    return torch.cat([model(X[i:i + 1000]).argmax(1) for i in range(0, len(X), 1000)])

## Scoring (do not modify)

In [ ]:
print(torch.cuda.get_device_name() if device == "cuda" else "CPU")
torch.cuda.synchronize()
t0 = time.perf_counter()
model = train()
torch.cuda.synchronize()
train_time = time.perf_counter() - t0
acc = (predict(model, test_X) == test_y).float().mean().item()
print(f"training time {train_time:.1f} s,  test accuracy = {acc:.4f}")

## Grading

Training time must be within 10 minutes (over the limit scores 0). Points are based on test accuracy:

* 94% or above: 30 points
* 92%: 25 points
* 90%: 20 points
* 85%: 12 points
* 80%: 6 points

(scores in between are linearly interpolated)